In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 500
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 150
mpl.rc("savefig", dpi=150)

from matplotlib.lines import Line2D

In [ ]:
data_folder = "../NYCDOE_data/"

filepath_SHSAT = data_folder + "SHSAT/"
filename_SHSAT = "_SHSAT_Gr 8_Match & No Match_Scrambled.csv"
filepath_demo = data_folder + "Demographics/"
filename_demo = "_June-Biog_PK-12_scrambled.csv"

In [ ]:
years = []
for x in np.arange(2005, 2017):
    years.append('%s-%s' % (str(x), str(x+1)[2:]))

In [ ]:
# school info and parameters
sch_info = {
    'B': {'name': 'Bronx High School of Science'},
    'T': {'name': 'Brooklyn Technical High School'},
    'R': {'name': 'Staten Island Technical High School'},
    'L': {'name': 'Brooklyn Latin'},
    'Q': {'name': 'Queens High School for Science at York'},
    'M': {'name': 'High School of Mathematics, Science and Engineering at City College'},
    'S': {'name': 'Stuyvesant High School'},
    'A': {'name': 'High School of American Studies at Lehman College'}
}

In [ ]:
def obtain_data_update_capacity(year, sch_info):
    ## load data
    df = pd.read_csv(filepath_SHSAT + year + filename_SHSAT, 
                     dtype={'student_id_scram': object, 'ets_feeder': object, 'booklet': object})\
        .fillna({'acceptance':''}).dropna(subset=['choice'])\
        .rename(columns={'vanguard_feeder': 'ets_feeder'})
    df = df[df['student_id_scram']!='.']
    
    ## capacity of schools
    # 1. find which school each student is admitted to
    df['acceptance'] = df.apply(lambda x: x['acceptance'] if x['acceptance']!='' 
                                else ''.join(x[['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8']].fillna('')), 
                                axis=1)
    df['accepted'] = df.apply(lambda x: ' ' if (x['acceptance'].find('A')==-1) 
                              else x['choice'][x['acceptance'].find('A')], axis=1)
    # 2. add capacity to dictionairy
    for sch in sch_info:
        sch_info[sch]['capacity'] = np.sum(df['accepted'] == sch)
    
    ## student classification
    # 1. demographic information
    demo_df = pd.read_csv(
        filepath_demo + year + filename_demo, 
        dtype={'student_id_scram': object, 'admit_code': object}
    ).drop_duplicates()
    df = pd.merge(df, demo_df[['student_id_scram', 'ell', 'poverty']], on='student_id_scram', how='left')
    
    # 2. middle school information
    sq_df = pd.read_csv(data_folder+"SchoolQuality/2017-18_School_Quality_Reports.csv")
    sq_df['ets_feeder'] = sq_df['DBN'].apply(lambda x: x[:2]+ x[3:])
    sq_df.drop_duplicates(subset=['ets_feeder'], inplace=True)
    df = pd.merge(df, sq_df[['ets_feeder', 'Economic Need Index']], on='ets_feeder', how='left')
    
    # 3. label each student as advantaged or disadvantaged
    df.fillna({'ell':0, 'poverty':0, 'Economic Need Index':0})
    df['disadvantaged'] = (((df['ell']==1) | (df['poverty']==1)) & (df['Economic Need Index']>.6)).astype(int)
    
    ## remove irrelevant columns
    df = df[['student_id_scram', 'total_scale_score', 'choice', 'disadvantaged']]
    
    ## lottery number or each students (minority with lower lottery number [higher priorities])
    n_adv = np.sum(df['disadvantaged']==0)
    n_dis = np.sum(df['disadvantaged']==1)
    np.random.seed(725)
    lottery_adv = list(np.random.choice(np.arange(n_dis, n_dis+n_adv), size=n_adv, replace=False))
    lottery_dis = list(np.random.choice(np.arange(n_dis), size=n_dis, replace=False))
    lottery = []
    for i in range(n_adv+n_dis):
        if df['disadvantaged'].iloc[i]==0:
            lottery.append(lottery_adv.pop())
        else:
            lottery.append(lottery_dis.pop())
    df.insert(df.shape[1], 'lottery', np.array(lottery))
    
    return df

In [ ]:
from scipy.optimize import minimize_scalar
import scipy.stats as stats

In [ ]:
def find_beta(df):
    # disadv
    sd = df[df.disadvantaged == 1].total_scale_score.values
    # adv
    sa = df[df.disadvantaged == 0].total_scale_score.values
    # multiplicative, W
    def wobj(beta):
        return stats.wasserstein_distance(sd/beta, sa)

    # additive, W
    def ff(beta):
        return stats.wasserstein_distance(sd+beta, sa)

    beta = minimize_scalar(wobj, (.5, .9, 1.1), tol=1e-10).x
    gamma = minimize_scalar(ff, (-200, 0, 300), tol=1e-10).x
    
    return beta, gamma

In [ ]:
def add_debiased_score(df):
    mean_std_01 = df.groupby(
        'disadvantaged'
    ).agg(
        {'total_scale_score': [np.mean, np.std]}
    ).to_numpy()
    beta, gamma = find_beta(df)
    df['score_debiased'] = df.apply(
        lambda x: x['total_scale_score']/beta
            if x['disadvantaged']==1 
            else x['total_scale_score'], 
        axis=1
    )
    df['true_potential_additive'] = df.apply(
        lambda x: x['total_scale_score']+gamma
            if x['disadvantaged']==1 
            else x['total_scale_score'], 
        axis=1
    )
    df = df.rename(columns={
        'total_scale_score': 'perceived_potential',
        'score_debiased': 'true_potential',
    })
    return df, beta, gamma

In [ ]:
for year in years:
    df = obtain_data_update_capacity(year, sch_info)
    df, beta, gamma = add_debiased_score(df)
    print(year, beta, gamma)

In [ ]:
year = years[-1]
df = obtain_data_update_capacity(year, sch_info)
df, beta, gamma = add_debiased_score(df)

In [ ]:
df.head()

In [ ]:
print(beta, 1/beta, gamma)

In [ ]:
plt.figure(figsize=(5, 3))
cmap = plt.get_cmap('Set2')
plt.hist(
    df[df['disadvantaged']==0]['perceived_potential'], 
    bins=18, 
    density=True, 
    alpha=.8, 
    color=cmap(0),  
    hatch='\\\\', 
    label='$G_1$ students'
)
plt.hist(
    df[df['disadvantaged']==1]['perceived_potential'], 
    bins=18, 
    density=True, 
    alpha=.4, 
    color=cmap(1), 
    hatch='//', 
    label='$G_2$ students'
)
plt.legend(handlelength=3, handleheight=2.5, fontsize=8)
plt.title('Distribution of SHSAT scores by group (%s)' % (year))
plt.savefig('new_beta/SHSAT_distribution_orig.png', bbox_inches='tight')

In [ ]:
plt.figure(figsize=(5, 3))
cmap = plt.get_cmap('Set2')
plt.hist(
    df[df['disadvantaged']==0]['true_potential'], 
    bins=18, 
    density=True, 
    alpha=.8, 
    color=cmap(0),  
    hatch='\\\\', 
    label='$G_1$ students'
)
plt.hist(
    df[df['disadvantaged']==1]['true_potential'], 
    bins=18, 
    density=True, 
    alpha=.4, 
    color=cmap(1), 
    hatch='//', 
    label='%.2f X $G_2$ students' % (1/beta)
)
plt.legend(handlelength=3, handleheight=2.5, fontsize=8)
plt.title('SHSAT scores under multiplicative shift (%s)' % (year))
plt.savefig('new_beta/SHSAT_distribution_scaled_multiplicative.png', bbox_inches='tight')

In [ ]:
gamma_unnormalized = 49
plt.figure(figsize=(5, 3))
cmap = plt.get_cmap('Set2')
plt.hist(
    df[df['disadvantaged']==0]['true_potential_additive'], 
    bins=18, 
    density=True, 
    alpha=.8, 
    color=cmap(0),  
    hatch='\\\\', 
    label='$G_1$ students'
)
plt.hist(
    df[df['disadvantaged']==1]['true_potential_additive'], 
    bins=18, 
    density=True, 
    alpha=.4, 
    color=cmap(1), 
    hatch='//', 
    label=f'$G_2$ + {gamma:.0f} students'
)
plt.legend(handlelength=3, handleheight=2.5, fontsize=8)
plt.title('SHSAT scores under additive shift (%s)' % (year))
plt.savefig('new_beta/SHSAT_distribution_scaled_additive.png', bbox_inches='tight')

In [ ]:
# G1
vals = df[df['disadvantaged']==0]['perceived_potential']
print(f"G1, {np.mean(vals)=}, {np.var(vals)=}, {np.std(vals)=}")
# G2
vals = df[df['disadvantaged']==1]['perceived_potential']
print(f"G2, {np.mean(vals)=}, {np.var(vals)=}, {np.std(vals)=}")
# G2 multiplicative debiased
vals = df[df['disadvantaged']==1]['true_potential']
print(f"G2 multiplicative debiased, {np.mean(vals)=}, {np.var(vals)=}, {np.std(vals)=}")
# G2 additive debiased
vals = df[df['disadvantaged']==1]['true_potential_additive']
print(f"G2 additive debiased, {np.mean(vals)=}, {np.var(vals)=}, {np.std(vals)=}")

In [ ]:
def get_assignment(df, sch_info, algo, low=0, high=750):
    # sort students in order of priority (high priority = high score & low lottery)
    if algo == 'PP':     # perceived potential
        df_sorted = df.sort_values(
            ['perceived_potential', 'lottery'], 
            ascending=[False, True]
        )
    if algo == 'TP':     # true potential
        df_sorted = df.sort_values(
            ['true_potential', 'lottery'], 
            ascending=[False, True]
        )
    if algo == 'IP':     # interventional potential
        df['potential_partially_revealed'] = df.apply(
            lambda x: x['true_potential'] 
                if low <= x['true_potential'] <= high
                else x['perceived_potential'],
            axis=1
        )
        df_sorted = df.sort_values(
            ['potential_partially_revealed', 'lottery'], 
            ascending=[False, True]
        )
        
    # initialize number of students admitted by each school and capacities
    # each schhol is divided into two: for Majority (M) and for minority (m)
    admit_dict = {}
    for sch in sch_info:
        admit_dict[sch] = {}
        admit_dict[sch]['capacity'] = sch_info[sch]['capacity']
        admit_dict[sch]['admit'] = 0

    # assign students to schools
    sch_assigned = [' ' for i in range(df_sorted.shape[0])]
    rk_assigned = np.zeros(df_sorted.shape[0], dtype=int)
    
    for i in range(df_sorted.shape[0]):
        # student information
        student = df_sorted.iloc[i,:]
        choice = list(student['choice'].replace(" ", ""))
        rk_assigned[i] = len(choice) + 1  # init to be if unassigned
        
        # expand his or her preference list (depending on the mechanism)
        preferences = [x for x in choice]
        
        # assign to the first school with seats remainning
        for sch in preferences:
            if admit_dict[sch]['admit'] < admit_dict[sch]['capacity']:
                admit_dict[sch]['admit'] = admit_dict[sch]['admit'] + 1
                sch_assigned[i] = sch
                rk_assigned[i] = choice.index(sch[0]) + 1
                break

    # return assignment
    df_sorted['sch_'+algo] = sch_assigned
    df_sorted['rk_'+algo] = rk_assigned
    return df_sorted.sort_index()[
        ['student_id_scram', 'sch_'+algo, 'rk_'+algo]
    ]

In [ ]:
additive_df = pd.DataFrame(df)

In [ ]:
additive_df["true_potential"] = additive_df["true_potential_additive"]
additive_df["perceived_potential"] = additive_df["perceived_potential"]

In [ ]:
additive_df_match = additive_df.merge(
    get_assignment(additive_df, sch_info, 'PP'),
    on='student_id_scram'
).merge(
    get_assignment(additive_df, sch_info, 'TP'),
    on='student_id_scram'
)
additive_lowest_score_admit = np.min(additive_df_match[additive_df_match['sch_PP']!=' ']['true_potential'])//5*5
additive_df_cohort = additive_df[additive_df['true_potential']>=additive_lowest_score_admit]
additive_p = np.sum(additive_df_cohort['disadvantaged']==1)/additive_df_cohort.shape[0]

In [ ]:
additive_normalized_true_scores = additive_df_cohort.true_potential.values / additive_lowest_score_admit

import scipy.stats as stats
additive_alpha = stats.pareto.fit(additive_normalized_true_scores, 1, floc=0, fscale=1)[0]
additive_alpha

In [ ]:
49/additive_lowest_score_admit

In [ ]:
additive_lowest_score_admit

In [ ]:
additive_p

In [ ]:
gamma

In [ ]:
beta

In [ ]:
df_match = df.merge(
    get_assignment(df, sch_info, 'PP'),
    on='student_id_scram'
).merge(
    get_assignment(df, sch_info, 'TP'),
    on='student_id_scram'
)
lowest_score_admit = np.min(df_match[df_match['sch_PP']!=' ']['true_potential'])//5*5
df_cohort = df[df['true_potential']>=lowest_score_admit]
p = np.sum(df_cohort['disadvantaged']==1)/df_cohort.shape[0]

In [ ]:
np.min(df_match[df_match['sch_PP']!=' ']['true_potential'])//5*5

In [ ]:
print(lowest_score_admit)

In [ ]:
p

In [ ]:
f"{p:.3f}"

In [ ]:
normalized_true_scores = df_cohort.true_potential.values / lowest_score_admit

import scipy.stats as stats
alpha = stats.pareto.fit(normalized_true_scores, 1, floc=0, fscale=1)[0]
alpha

In [ ]:
beta

In [ ]:
def plot_displacement(df_match, low, high, plotting=False):
    df_match['displacement'] = \
        df_match['rk_PP'] - df_match['rk_TP']
    if not low==None:
        df_match = df_match.merge(
            get_assignment(df, sch_info, 'IP', low, high),
            on='student_id_scram'
        )
        df_match['displacement_with_intervention'] = \
            df_match['rk_IP'] - df_match['rk_TP']
    if not plotting:
        return df_match
    
    plt.figure(figsize=(10,4))
    plt.plot(
        df_match[df_match['disadvantaged']==1]['true_potential'], 
        df_match[df_match['disadvantaged']==1]['displacement'], 
        'bh', markersize=3, alpha=.2
    )
    plt.plot(
        df_match[df_match['disadvantaged']==0]['true_potential'], 
        df_match[df_match['disadvantaged']==0]['displacement'], 
        'mo', markersize=3, alpha=.2
    )
    
    if not low==None:
        plt.plot(
            df_match[
                (df_match['disadvantaged']==1) & 
                (df_match['displacement']<df_match['displacement_with_intervention'])
            ]['true_potential'], 
            df_match[
                (df_match['disadvantaged']==1) & 
                (df_match['displacement']<df_match['displacement_with_intervention'])
            ]['displacement_with_intervention'],
            'gd', markersize=4, alpha=.2, 
        )
        plt.plot(
            df_match[
                (df_match['disadvantaged']==1) & 
                (df_match['displacement']>df_match['displacement_with_intervention'])
            ]['true_potential'], 
            df_match[
                (df_match['disadvantaged']==1) & 
                (df_match['displacement']>df_match['displacement_with_intervention'])
            ]['displacement_with_intervention'],
            'rd', markersize=4, alpha=.2, 
            label='$G_2$ with smaller displacement under intervention'
        )
        plt.axvline(low, linestyle='--', color='gray', label='debiased range')
        plt.axvline(high, linestyle='--', color='gray')
    plt.xlim([lowest_score_admit, 700])
    plt.ylabel('displacement')
    plt.xlabel('true potential')
    plt.legend(
        handles = [
            Line2D(
                [0], [0], markerfacecolor='b', marker='h', markersize=6, color='w',
                label='$G_2$ (no intervention)'
            ),
            Line2D(
                [0], [0], markerfacecolor='m', marker='o', markersize=6, color='w',
                label='$G_1$ (no intervention)'
            ),
            Line2D(
                [0], [0], markerfacecolor='g', marker='d', markersize=8, color='w',
                label='$G_2$ with larger displacement under intervention'
            ),
            Line2D(
                [0], [0], markerfacecolor='r', marker='d', markersize=8, color='w',
                label='$G_2$ with smaller displacement under intervention'
            )
            ]
    )
    return df_match

In [ ]:
_ = plot_displacement(df_match, None, 350, plotting=True)

In [ ]:
_ = plot_displacement(additive_df_match, None, 350, plotting=True)

In [ ]:
additive_df_match_IP = plot_displacement(additive_df_match, 525, 550, plotting=True)

In [ ]:
df_match_IP = plot_displacement(df_match, 525, 550, plotting=True)
plt.savefig("new_beta/shsat-doe-displacement.png", bbox_inches="tight")

In [ ]:
df_temp = df_match_IP[
    (df_match_IP['disadvantaged']==1) & 
    (df_match_IP['displacement']<df_match_IP['displacement_with_intervention'])
]
df_temp_better = df_match_IP[
    (df_match_IP['disadvantaged']==1) & 
    (df_match_IP['displacement']>df_match_IP['displacement_with_intervention'])
]

In [ ]:
df_temp_better.shape[0]

In [ ]:
np.mean(df_temp['displacement_with_intervention']-df_temp['displacement'])

In [ ]:
df_match_IP[(df_match_IP['disadvantaged']==1)].shape[0]

<h3 style="color:red"> optimal intervention (PAUC only)</h3>

In [ ]:
c = 0.1
step = 5
TPs = df_cohort['true_potential']
TPs_dis = df_cohort[df_cohort.disadvantaged == 1].true_potential
breakpoints_lft = [np.percentile(TPs_dis, x) for x in np.arange(0,101-100*c,step)]
breakpoints_rgt = [np.percentile(TPs_dis, x) for x in np.arange(100*c,101,step)]

In [ ]:
df = df_match.copy()

In [ ]:
pauc_arr = []
for i in np.arange(len(breakpoints_lft)):
    df_match = plot_displacement(df, breakpoints_lft[i], breakpoints_rgt[i])
    displacements = df_match['displacement_with_intervention']
    pauc_arr.append(np.mean(displacements[displacements>0]))
    print('voucher to %3d to %3d percentile || ' % (i*step, i*step+100*c), 
          'scores [%6.2f, %6.2f] || ' % (breakpoints_lft[i], breakpoints_rgt[i]),
          'PAUC', np.mean(displacements[displacements>0])
    )
# no intervetion
displacements = df_match['displacement']
print('no intervention', np.mean(displacements[displacements>0]))

In [ ]:
c = 0.4
step = 5
TPs = df_cohort['true_potential']
breakpoints_lft = [np.percentile(TPs_dis, x) for x in np.arange(0,101-100*c,step)]
breakpoints_rgt = [np.percentile(TPs_dis, x) for x in np.arange(100*c,101,step)]

In [ ]:
pauc_arr = []
for i in np.arange(len(breakpoints_lft)):
    df_match = plot_displacement(df, breakpoints_lft[i], breakpoints_rgt[i])
    displacements = df_match['displacement_with_intervention']
    pauc_arr.append(np.mean(displacements[displacements>0]))
    print('voucher to %3d to %3d percentile || ' % (i*step, i*step+100*c), 
          'scores [%6.2f, %6.2f] || ' % (breakpoints_lft[i], breakpoints_rgt[i]),
          'PAUC', np.mean(displacements[displacements>0])
    )
# no intervetion
displacements = df_match['displacement']
print('no intervention', np.mean(displacements[displacements>0]))

<h3 style="color:red"> ADDITIVE optimal intervention (PAUC only)</h3>

In [ ]:
c = 0.1
step = 1
TPs = additive_df_cohort['true_potential']
TPs_dis = additive_df_cohort[df_cohort.disadvantaged == 1].true_potential
breakpoints_lft = [np.percentile(TPs_dis, x) for x in np.arange(0,101-100*c,step)]
breakpoints_rgt = [np.percentile(TPs_dis, x) for x in np.arange(100*c,101,step)]

# breakpoints_lft = [np.percentile(TPs_dis, x) for x in np.arange(50,55,step)]
# breakpoints_rgt = [np.percentile(TPs_dis, x) for x in np.arange(50+c*100,55+c*100,step)]

additive_df = additive_df_match.copy()

pauc_arr = []
for i in np.arange(len(breakpoints_lft)):
    additive_df_match2 = plot_displacement(additive_df, breakpoints_lft[i], breakpoints_rgt[i])
    displacements = additive_df_match2['displacement_with_intervention']
    pauc_arr.append(np.mean(displacements[displacements>0]))
    print('voucher to %3d to %3d percentile || ' % (i*step, i*step+100*c), 
          'scores [%6.2f, %6.2f] || ' % (breakpoints_lft[i], breakpoints_rgt[i]),
          'PAUC', np.mean(displacements[displacements>0])
    )
# no intervetion
displacements = additive_df_match['displacement']
print('no intervention', np.mean(displacements[displacements>0]))

In [ ]:
c = 0.4
step = 1
TPs = additive_df_cohort['true_potential']
breakpoints_lft = [np.percentile(TPs_dis, x) for x in np.arange(0,101-100*c,step)]
breakpoints_rgt = [np.percentile(TPs_dis, x) for x in np.arange(100*c,101,step)]

In [ ]:
additive_df = additive_df_match.copy()

In [ ]:
pauc_arr = []
for i in np.arange(len(breakpoints_lft)):
    additive_df_match2 = plot_displacement(additive_df, breakpoints_lft[i], breakpoints_rgt[i])
    displacements = additive_df_match2['displacement_with_intervention']
    pauc_arr.append(np.mean(displacements[displacements>0]))
    print('voucher to %3d to %3d percentile || ' % (i*step, i*step+100*c), 
          'scores [%6.2f, %6.2f] || ' % (breakpoints_lft[i], breakpoints_rgt[i]),
          'PAUC', np.mean(displacements[displacements>0])
    )
# no intervetion
displacements = additive_df_match['displacement']
print('no intervention', np.mean(displacements[displacements>0]))

<h3 style="color:red"> Pareto distribution fitting </h3>

In [ ]:
import scipy.stats as stats

In [ ]:
scores_top_normalized = df_cohort['true_potential']/lowest_score_admit

In [ ]:
scores_top_normalized

In [ ]:
alpha = stats.pareto.fit(scores_top_normalized, 1, floc=0, fscale=1)[0]

In [ ]:
alpha

In [ ]:
plt.figure(figsize=(5,3))
plt.hist(scores_top_normalized*lowest_score_admit, bins=15, 
         density=True, alpha=.6, label='Data')
xs = np.arange(1.05, 1.55,.05)
plt.plot(xs*lowest_score_admit, alpha/xs**(alpha+1)/lowest_score_admit, label='Fitted')
plt.xlabel('Estimated true potential')
plt.ylabel('Density')
plt.legend()
plt.savefig("new_beta/pareto-fitting-shsat.png", bbox_inches="tight")

<h3 style="color:red"> Plugging in the value </h3>

In [ ]:
print(
    'alpah=', alpha, '\n',
    'beta=', beta, '\n',
    'p=', p, '\n',
)

In [ ]:
print('assumption checking: ') 
p<1-beta**alpha

In [ ]:
c_boundary_sm_lg = (1-p)*(1-beta**alpha)/(1-p+1-beta**alpha)
c_boundary_sm_lg

In [ ]:
c=0.1
ba = beta**alpha
pba = p*ba
invba = 1/ba
if c >= c_boundary_sm_lg:
#     Z1_star = (((1-p)+(1/(beta**alpha)-1)*c)/(1/(beta**alpha)-p))**(-1/alpha)
#     Z2_star = ((1-p)*(1-c)/(1/(beta**alpha)-p))**(-1/alpha)
    Z1_star = ((1-p)*(1-c)/(pba+invba-2*p)+c)**(-1/alpha)
    Z2_star = ((1-p)*(1-c)/(pba+invba-2*p))**(-1/alpha)
    print('large c: Z1=%8.3f and Z2=%8.3f' % (Z1_star*lowest_score_admit, Z2_star*lowest_score_admit))
else:
#     Z1_star = (((1-p-c)*(beta**alpha))/(1-p)+c)**(-1/alpha)
#     Z2_star = (((1-p-c)*(beta**alpha))/(1-p))**(-1/alpha)
    Z1_star = (((pba-1)*c+(1-p))/((1-p)*invba)+c)**(-1/alpha)
    Z2_star = (((pba-1)*c+(1-p))/((1-p)*invba))**(-1/alpha)
    print('small c: Z1=%8.3f and Z2=%8.3f' % (Z1_star*lowest_score_admit, Z2_star*lowest_score_admit))
    

In [ ]:
c=0.4 
ba = beta**alpha
pba = p*ba
invba = 1/ba
if c >= c_boundary_sm_lg:
#     Z1_star = (((1-p)+(1/(beta**alpha)-1)*c)/(1/(beta**alpha)-p))**(-1/alpha)
#     Z2_star = ((1-p)*(1-c)/(1/(beta**alpha)-p))**(-1/alpha)
    Z1_star = ((1-p)*(1-c)/(pba+invba-2*p)+c)**(-1/alpha)
    Z2_star = ((1-p)*(1-c)/(pba+invba-2*p))**(-1/alpha)
    print('large c: Z1=%8.3f and Z2=%8.3f' % (Z1_star*lowest_score_admit, Z2_star*lowest_score_admit))
else:
#     Z1_star = (((1-p-c)*(beta**alpha))/(1-p)+c)**(-1/alpha)
#     Z2_star = (((1-p-c)*(beta**alpha))/(1-p))**(-1/alpha)
    Z1_star = (((pba-1)*c+(1-p))/((1-p)*invba)+c)**(-1/alpha)
    Z2_star = (((pba-1)*c+(1-p))/((1-p)*invba))**(-1/alpha)
    print('small c: Z1=%8.3f and Z2=%8.3f' % (Z1_star*lowest_score_admit, Z2_star*lowest_score_admit))
    

In [ ]:
# pauc no intervention
1/2*(1-p)*(1-beta**alpha)*8